# Exercise session 0: Introduction to Google Colab, NumPy, Matplotlib and SciPy

In [ ]:
print("Hello World")

## Introduction to Google Colab

Colab notebooks (.ipynb) run on Google's managed Python runtime.

- Add a text cell: click "+ Text" (toolbar or between cells)
- Add a code cell: click "+ Code"
- Run a cell: press Shift+Enter or click the Run ▶ button on the cell
- Insert a cell between blocks: hover between cells to reveal the "+" insert control
- Keep narrative in Markdown cells; put computations in Code cells

### Additional notes

- Python organizes functionality into modules and packages. Use the `import` statement to bring names into scope (e.g., `import numpy as np`).
- Google Colab ships with many packages preinstalled. If an import fails with `ModuleNotFoundError`, the package is not available in the current runtime; install it in a notebook cell (e.g., `%pip install <package>`) and re-run the import.
- Prefer `%pip` over shell-style `!pip` inside notebooks; it ensures installation targets the active kernel environment.
- A runtime restart is usually not required after installation. If the module is still unavailable, use Runtime > Restart session and re-execute the relevant cells.

### Build your own module

- Code defined in a notebook is scoped to that notebook. To reuse utilities across multiple notebooks, put them in a Python module (`.py` file) and import them.
- Example module (`utils.py`):

```python
# utils.py
def print_hello_world():
    # print("I've been updated")  # commented: skipped at runtime
    print("Hello World!")
```

- Usage from a notebook in the same folder:

```python
from utils import print_hello_world
print_hello_world()
```

- If you call the function without importing it, Python raises a `NameError`:

```python
# NameError: name 'print_hello_world' is not defined
print_hello_world()
```
To be able to import the module, you must first upload it to your Colab sesion. Click the folder icon on the left, click the upload icon, and upload `utils.py` (which can be found on Toledo).

In [1]:
from utils import print_hello_world
print_hello_world()

# once the module is imported it will be cached in memory, so a change in the module file will not be reflected even if you re-import it
# try to change the function removing the comment in utils.py and re-run this cell

Hello, world!


In [ ]:
# To force reload of the module you need to restart the Google Colab runtime or reload it using importlib
import importlib
import utils
importlib.reload(utils)
from utils import print_hello_world

print_hello_world()

## Introduction to NumPy

NumPy is the fundamental package for numerical computing in Python: n-dimensional arrays, vectorized operations, linear algebra, and random number generation. We'll use it for loading, transforming, and feature extraction. To import NumPy, run the following code block.

In [7]:
import numpy as np

Let's create our very first toy dataset using NumPy. We will use `np.linspace()` for this. Another option is the function `np.arange()`. Insert a code block below this text block and look up the documentation of one of these functions. To find the documentation, you can either look it up on the internet, or you can run `help(np.linspace)` or `help(np.arange)`.

Let's use this function to create an array `a`, ranging from 0 to 10 (inclusive), containing 1001 elements. Print and inpect `a`. Verify length (hint: use `len()`, `np.shape` or `np.size` for this; look up the documentation of these functions if needed). Check its minimum (`np.min()`), maximum (`np.max()`) and average (`np.mean()`).

Create a new array, `b`, containing the sine of all values in `a`. Hint: use `np.sin()` for this. Look up its documentation if needed.

Finally, create a third array, `c`, being the cosine values of `a`. Can you guess which NumPy function you could use for this?

Download the data `ex0  ` from Toledo. To be able to read in data, we first need to add the data to the current Colab session. In the sidebar on the left of the Colab window, press the `folder` icon. Next, press the `upload` icon. Browse to the correct location on your computer, and select the correct file to upload. Read in the data using `np.load()`. Inspect the data array. What is its shape, maximum, minimum, average?

### NumPy numeric types (dtypes)

NumPy implements multiple numeric dtypes with explicit width/precision. We will be mostly interested in:

- Integer : `int8`, `int16`, `int32`, `int64`.
- Floating point: `float16`, `float32` (single precision), `float64` (double precision).
- Complex: `complex64` (2×float32), `complex128` (2×float64).
- Bool: `bool_`.

Note:
- The numpy dtype determines the amount of memory occupied from each number. In general, the higher the precision, the higher the memory occupation. Therefore, be careful to use 64bit precision by default when dealing with a huge amount of data.

In [ ]:
# Dtypes and memory
x32 = np.zeros((1000, 1000), dtype=np.float32)
x64 = np.zeros((1000, 1000), dtype=np.float64)
print(x32.dtype, x32.nbytes/1e6, "MB")
print(x64.dtype, x64.nbytes/1e6, "MB")


# Note if you create a numpy dtype int you cannot assign a float value to it, it will be truncated
xint = np.zeros((10,), dtype=np.int32)
print(xint)
xint[0] = 1.5
print(xint)

### Vectorization

NumPy provides vectorized operations that act on whole arrays at once. Whenever you find yourself writing a Python for-loop over array elements, check whether a NumPy ufunc or built-in routine exists—it will be faster and more concise by default.

- Use ufuncs (e.g., `np.sin`, `np.exp`, `np.sqrt`) and broadcasting instead of element-wise loops.
- Prefer array expressions (`a + b`, `a * b`) over manual iteration.
- If a loop is unavoidable, preallocate the output array to avoid repeated allocations.
- In notebooks, use `%timeit` to benchmark vectorized code vs. loops.

In [ ]:
# Vectorization vs loop
N = 2000000
v = np.linspace(0, 2*np.pi, N)
%timeit np.sin(v) # vectorized  operation


def sin_loop(vec):
    out = np.empty_like(vec)
    for i in range(vec.size):
        out[i] = np.sin(vec[i])
    return out
%timeit sin_loop(v) # for loop operation

### NumPy seeding and reproducibility

Many operations rely on pseudo-random numbers. To obtain reproducible results across runs, fix a seed and use NumPy’s Generator API.

- Prefer the modern Generator: `rng = np.random.default_rng(42)` and then use `rng.normal`, `rng.integers`, etc.
- Initialize the RNG near the start of your notebook/script and reuse the same `rng` object for all random draws.
- Many libraries accept a `random_state` or `seed` argument—set it to a fixed integer to make those components reproducible.
- Avoid `np.random.seed(...)` in library code because it mutates global state; it is acceptable for quick notebooks, but the Generator API is safer and more explicit.
- If you want different but controlled runs, vary the seed (e.g., 0, 1, 2, …) and log it with your results.

In [ ]:
rng = np.random.default_rng(42)  # reseed to reproduce
r1 = rng.normal(size=5) # creates 5 elements drawn from a normal distribution

rng = np.random.default_rng(42)  # reseed to reproduce
r2 = rng.normal(size=5)

print(np.allclose(r1, r2))
print( r1, r2 )

## Introduction to Matplotlib

Matplotlib is a plotting library that we will use to make visualizations. Run the following code block to import this package.

In [ ]:
import matplotlib.pyplot as plt

The basic skeleton for making a Matplotlib visulization is provided below. Use `help(plt.plot())` for parameters. Complete the figure with a title, x-axis label, and y-axis label. Hint: use `ax.set_xlabel()`, `ax.set_ylabel()`, `ax.set_title()`.

In [ ]:
fig, ax = plt.subplots()
ax.plot(a,b)
# ...

Plot the cosine values (array `c`) on the same axes. Add a legend, mapping lines to labels. Hint: set `label` in `ax.plot()`; then call `plt.legend()`.

In [ ]:
fig, ax = plt.subplots()
ax.plot(a,b) # adapt this line, use the "label" argument
ax.plot(a,c) # adapt this line, use the "label" argument
# ...

Now, draw the sine and cosine in two separate subplots. Complete the skeleton below for this. Use `ax1` for plotting the sine (array `b`), and `ax2` for plotting the cosine (array `c`). Add axis labels and subtitles.

In [ ]:
fig, [ax1,ax2] = plt.subplots(2,1)
fig.suptitle("My third and fourth plots")
# ax1
# ...

# ax2
# ...

fig.tight_layout()

Now write your own code block to plot the data from `ex0.npy` that you loaded in the previous part. Can you guess what this signal could be?

## Introduction to SciPy

SciPy builds on NumPy with algorithms for optimization, signal processing, statistics, interpolation, and more. For now, we’ll focus on normalization with `scipy.stats.zscore` and leave filtering/spectral analysis for later lessons.





In [ ]:
import scipy

Use the function `scipy.stats.zscore()` to normalize the data that was loaded previously. Inspect the effect of normalization: what is now the minimum, maximum, and average value?

Plot the normalized signal.

## Introduction to type hints

Python is dynamically typed at runtime, but you can add static type hints to improve readability and enable tooling (linters, type checkers). Below, we demonstrate dynamic typing, where a variable can change type throughout a program without any issues.

In [ ]:
def helloWorld():
  a = "hello"
  print(a)
  print(type(a))

  a = 1
  print(a)
  print(type(a))

helloWorld()

However, you can annotate functions with type hints. Argument types follow a colon; the return type follows `->`. Tools can use these hints to detect mismatches before running your code.

In [ ]:
def helloWorld(provideText: bool) -> None:
  if provideText:
    a = "hello"
    print(a)
    print(type(a))

  else:
    a = 1
    print(a)
    print(type(a))

helloWorld(provideText=True)

In [ ]:
helloWorld(provideText=False)

Type hints improve readability but do not enforce types at runtime. If you pass a string to `provideText` (expected `bool`), Python will treat non-empty strings as truthy in conditionals. Test this out. Can you explain the output?

To catch type errors, you could enable static checking (e.g., in your IDE) or run a type checker such as `mypy` on your code. In Colab, you can also rely on editor diagnostics to flag mismatches (`Tools` > `Settings` > `Editor` > `Code diagnostics` > `Syntax and type checking`). Check what happens if you do this.

Note: Type hints are optional but recommended for larger projects and shared code. We'll use them in templates (especially around NumPy arrays) to improve readability and intent.